Import Libraries

In [ ]:
import cv2 
import mediapipe as mp
import math
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
import joblib


Load Dataset

In [ ]:
dataset_path = "SignAlphaSet"
data=None
path=[]
label=[]
letters = os.listdir(dataset_path)
for letter in letters:
  letter_path=os.path.join(dataset_path,letter)
  data=os.listdir(letter_path)
  for i in data:
   path.append(os.path.join(letter_path,i))
   label.append(letter)
print(path)
print(label)


Extract Hand Landmarks

In [ ]:
mp_hand=mp.solutions.hands
hands=mp_hand.Hands(
  max_num_hands=1,
  min_detection_confidence=0.5,
  min_tracking_confidence=0.5,
)
fingers = []

for img_path, label_img in zip(path, label):

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:

        for landmark in results.multi_hand_landmarks:

            image_data = normalize_landmarks(landmark)

            fingers.append([*image_data, label_img])

print(len(fingers[0]))
print(len(fingers))

Normalize Landmarks

In [ ]:
def normalize_landmarks(landmark):
    points = landmark.landmark
    wrist = points[0]
    relative_points = []

    for point in points:
        x = point.x - wrist.x
        y = point.y - wrist.y
        z = point.z - wrist.z
        relative_points.append([x, y, z])

    scale = max(
        math.sqrt(x**2 + y**2)
        for x, y, z in relative_points
    )

    if scale < 1e-6:
        return [0] * 63

    features = []
    for x, y, z in relative_points:
        features.extend([
            x / scale,
            y / scale,
            z / scale
        ])
        
    return features

In [ ]:
columns = []

for i in range(21):
    columns.extend([f"{i}_x", f"{i}_y", f"{i}_z"])

columns.append("Label")

df = pd.DataFrame(fingers, columns=columns)

In [ ]:
df.to_csv("hand_landmarks2.csv", index=False)

In [ ]:
data=pd.read_csv("hand_landmarks2.csv")

In [ ]:
x_model = data.drop(columns=["Label"])
y_model = data["Label"]

Split Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_model, y_model,
    test_size=0.2,
    random_state=42,
   
)

Train Random Forest Model

In [ ]:
model =  RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
joblib.dump(model, "sign_language_model.pkl")

Real-Time Sign Language Recognition

If you don't have the dataset, you can use the pre-trained model (`sign_language_model.pkl`) and run the cells starting from this section.

In [ ]:
model = joblib.load("sign_language_model.pkl")

In [ ]:
cap=cv2.VideoCapture(0)

#draw landmarks
mp_draw=mp.solutions.drawing_utils
hand_style=mp_draw.DrawingSpec(
  color=(60,170,80),
  circle_radius=2
)
conection_style=mp_draw.DrawingSpec(
  color=(60,210,90),
  thickness=1
)

while True :
  ret,frame=cap.read()
  if not ret: 
    break
  frame=cv2.flip(frame,1)
  frame_rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
  results=hands.process(frame_rgb)
  if results.multi_hand_landmarks:
    for landmark in results.multi_hand_landmarks:
            features = normalize_landmarks(landmark)
            if len(features) == 63:

              data = pd.DataFrame(
                  [features],
                  columns=x_model.columns
              )

              probabilities = model.predict_proba(data)[0]
              best_index = probabilities.argmax()
              prediction = model.classes_[best_index]
              print(prediction)
            

            mp_draw.draw_landmarks(
                frame,
                landmark,
                mp_hand.HAND_CONNECTIONS,
                hand_style,
                conection_style)
  cv2.imshow("frame",frame)
  if cv2.waitKey(1)==ord('q'):
        break
cap.release()
cv2.destroyAllWindows()